# model_experiment_ARIMA_SARIMA

მიზანი: classical statistical approach-ის გარჩევა და პატარა ექსპერიმენტი. Walmart dataset-ში ათასობით `Store + Dept` time series გვაქვს, ამიტომ ARIMA/SARIMA ყველა სერიაზე პრაქტიკულად ნელია. ამიტომ აქ გვაქვს:

1. Seasonal naive baseline full dataset-ზე.
2. SARIMA მხოლოდ რამდენიმე representative/top series-ზე.


In [1]:
# Run once if packages are missing:
# !pip install -r requirements.txt

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import joblib
import wandb

from src.config import WANDB_ENTITY, WANDB_PROJECT, MODELS_DIR, SUBMISSIONS_DIR, VALIDATION_FOLDS, VALIDATION_WEEKS, RANDOM_STATE
from src.data_loading import load_raw_data, describe_dataframes, make_submission_frame
from src.validation import evaluate_time_folds, summarize_cv
from src.metrics import wmae
from src.wandb_utils import safe_wandb_init, save_and_log_model, log_dataframe_as_artifact

pd.set_option('display.max_columns', 100)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)
print('W&B entity:', WANDB_ENTITY)
print('W&B project:', WANDB_PROJECT)

from src.classical import SeasonalNaiveForecaster, evaluate_sarima_selected_series, top_series

W&B entity: gchal22-free-university-of-tbilisi-
W&B project: store_sales_forecast


In [3]:
wandb.login()
data = load_raw_data()
train = data['train'].copy()
test = data['test'].copy()
sample_submission = data['sample_submission'].copy()
train['Date'] = pd.to_datetime(train['Date'])

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\West\_netrc.
wandb: Currently logged in as: gchal22 (gchal22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Seasonal naive baseline

Baseline: იგივე Store+Dept-ის 52 კვირის წინანდელი გაყიდვა. ეს retail weekly forecasting-ში baseline-ია.

In [4]:
from src.validation import make_time_folds

folds = make_time_folds(train, n_folds=3, validation_weeks=8)
rows = []
for i, (_, train_end, valid_start, valid_end) in enumerate(folds, start=1):
    fold_train = train[train['Date'] <= train_end].copy()
    fold_valid = train[(train['Date'] >= valid_start) & (train['Date'] <= valid_end)].copy()
    model = SeasonalNaiveForecaster().fit(fold_train)
    preds = model.predict(fold_valid.drop(columns=['Weekly_Sales']))
    score = wmae(fold_valid['Weekly_Sales'], preds, fold_valid['IsHoliday'])
    rows.append({'fold': i, 'train_end': train_end, 'valid_start': valid_start, 'valid_end': valid_end, 'wmae': score})

seasonal_naive_cv = pd.DataFrame(rows)
seasonal_naive_cv

,fold,train_end,valid_start,valid_end,wmae
0,1,2012-05-11,2012-05-18,2012-07-06,1777.644077
1,2,2012-07-06,2012-07-13,2012-08-31,1690.629814
2,3,2012-08-31,2012-09-07,2012-10-26,1680.421016


In [5]:
with safe_wandb_init('SeasonalNaive_CV', 'Classical', 'cross_validation') as run:
    wandb.log({
        'cv_wmae_mean': float(seasonal_naive_cv['wmae'].mean()),
        'cv_wmae_std': float(seasonal_naive_cv['wmae'].std(ddof=0)),
    })
    wandb.log({'seasonal_naive_cv': wandb.Table(dataframe=seasonal_naive_cv)})
    log_dataframe_as_artifact(run, seasonal_naive_cv, 'seasonal-naive-cv-results', 'validation_results', 'reports/seasonal_naive_cv_results.csv')

cv_wmae_mean,▁
cv_wmae_std,▁
cv_wmae_mean,1716.23164
cv_wmae_std,43.62469


## SARIMA selected series

In [6]:
N_SERIES = 3
series_keys = top_series(train, n=N_SERIES)
series_keys

[(14, 92), (2, 92), (20, 92)]

In [7]:
with safe_wandb_init(
    'SARIMA_Selected_Series',
    'Classical',
    'selected_series_experiment',
    config={'n_series': N_SERIES, 'order': '(1,1,1)', 'seasonal_order': '(1,0,1,52)'},
) as run:
    sarima_results = evaluate_sarima_selected_series(train, series_keys=series_keys, validation_weeks=8)
    wandb.log({'sarima_results': wandb.Table(dataframe=sarima_results)})
    if sarima_results['wmae'].notna().any():
        wandb.log({'sarima_selected_series_mean_wmae': float(sarima_results['wmae'].mean())})
    log_dataframe_as_artifact(run, sarima_results, 'sarima-selected-series-results', 'validation_results', 'reports/sarima_selected_series_results.csv')

sarima_results

sarima_selected_series_mean_wmae,▁
sarima_selected_series_mean_wmae,8931.07443


,store,dept,method,wmae,status
0,14,92,SARIMA,15658.048720,ok
1,2,92,SARIMA,7019.289372,ok
2,20,92,SARIMA,4115.885191,ok


## Optional seasonal naive submission

In [8]:
seasonal_model = SeasonalNaiveForecaster().fit(train)
preds = seasonal_model.predict(test)
submission = make_submission_frame(test, preds, sample_submission)
submission_path = SUBMISSIONS_DIR / 'submission_seasonal_naive.csv'
submission.to_csv(submission_path, index=False)
submission.head()

,Id,Weekly_Sales
0,1_1_2012-11-02,39886.06
1,1_1_2012-11-09,18689.54
2,1_1_2012-11-16,19050.66
3,1_1_2012-11-23,20911.25
4,1_1_2012-11-30,25293.49
